In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [7]:
def generate_random_haar(n: int, rng = np.random.default_rng()):

    """
    This function generates an nxn random Haar 
    matrix according to the article by Mezzadri. (QR decomposition)
    """

    Z = (rng.normal(size=(n, n)) + 1j * rng.normal(size=(n, n))) / np.sqrt(2.0) # random initializing
    Q, R = np.linalg.qr(Z) # QR decomposition step
    d = np.diag(R)
    ph = d / np.abs(d)
    Q = Q * np.conj(ph)

    return Q

def gauge_unfixing(vector):
  """
  In general when you perform a linalg algorithm function there's some bias on the eigvec phase,
  this should manage the issue by defining a random phase for each vector
  
  """
  ɸ = np.random.uniform(0, 2*np.pi)
  return vector * np.exp(1j * ɸ)


def build_perm(Sk1, Sk2):
  """Build permanent form"""
  return np.outer(Sk1, Sk2) + np.outer(Sk2, Sk1)



In [48]:
def align_sigma_vector(eigvals, eigvecs):
    """
    This function aligns an array of +-1 value (our phases) to the sign of 
    the real (and imaginary) part of the maximum eigenvalue eigenvector.

    Args:
        eigvals (np.ndarray): eigenvalues array.
        eigvecs (np.ndarray): eigenvectors array.

    Returns:
        np.ndarray: aligned phases vector (+1 o -1) for real alignment.
        np.ndarray: aligned phases vector (+1 o -1) for imaginary alignment
        np.ndarray: selected maximum eigvec
        complex: max eigval.
    """
    # find the index relative to the maximal eigvalue in modulus
    selected_idx = np.argmax(np.sqrt(eigvals.real**2+eigvals.imag**2))

    selected_eigenvalue = eigvals[selected_idx]
    selected_eigenvector = eigvecs[:, selected_idx]
    selected_eigenvector = gauge_unfixing(eigvecs[:, selected_idx]) # phase freedom randomized

    # extract the alignment part
    target_part = selected_eigenvector.real

    # create the sigma vector (phases)
    sigma_r = np.where(target_part >= 0, 1, -1)

    target_part = selected_eigenvector.imag 

    # create the sigma vector (phases)
    sigma_i = np.where(target_part >= 0, 1, -1)

    return sigma_r, sigma_i, selected_eigenvector, selected_eigenvalue


The overlap is defined as:
$$\mathbb M^{(k_1,k_2)} = \frac{2\sqrt{2}}{\sqrt{\mu(\vec k)}}  m^{(k_1)} m^{(k_2)} - \frac{2(\sqrt{2}-1)}{ \sqrt{2} M} \delta_{k_1,k_2} = \left(1-\delta_{k_1,k_2}\right) 2\sqrt{2}\,  m^{(k_1)} m^{(k_2)} + \delta_{k_1,k_2}\left[
2\left(m^{(k_1)}\right)^2-
\frac{2-\sqrt{2}}{ M} 
\right]$$
where $$m^{(k)} \equiv \frac{1}{\sqrt{M}}\sum_{x=1}^M S_{k,x}\, \sigma_x \quad ; \quad \max_{M\gg 1} m^{(k)}=O(1)$$

In [49]:
def run_experiment(S, k1, k2):
    """
    This function is an experiment run:
    fixing a scattering matrix and the two measuring modes k1 and k2
    you get the max eigvec and eigval, the best sigmas i.e. the one 
    aligned respectively to the real and the imaginary part, and the
    corresponding overlap considering the two configurations
    """
    Sk1 = S[k1,:]
    Sk2 = S[k2,:]
    M = np.size(Sk1) # mode number 
    𝕏 = build_perm(Sk1, Sk2)
    eigvals_perm, eigvecs_perm = np.linalg.eig(𝕏) # linearizes the matrix

    threshold = 1e-14
    rounded_eigvals_perm = np.copy(eigvals_perm)

    # Set tiny imaginary parts to zero
    rounded_eigvals_perm.imag[np.abs(rounded_eigvals_perm.imag) < threshold] = 0.0

    # Set tiny real parts to zero
    rounded_eigvals_perm.real[np.abs(rounded_eigvals_perm.real) < threshold] = 0.0

    # compute best sigmas choosing both real and imaginary alignment

    sigma_real, sigma_imaginary, selected_eigenvector, selected_eigenvalue = align_sigma_vector(eigvals_perm, eigvecs_perm)

    if (k1 != k2): 

        𝕄_real = (2*np.sqrt(2)/M)*np.dot(Sk1, sigma_real)*np.dot(Sk2, sigma_real)

        𝕄_imaginary = (2*np.sqrt(2)/M)*np.dot(Sk1, sigma_imaginary)*np.dot(Sk2, sigma_imaginary)

    else: # bunched configuration

        𝕄_real = (2/M)*np.dot(Sk1, sigma_real)**2 - (2 - np.sqrt(2))/M

        𝕄_imaginary = (2/M)*np.dot(Sk1, sigma_imaginary)**2 - (2 - np.sqrt(2))/M

    return sigma_real, sigma_imaginary, 𝕄_real, 𝕄_imaginary, selected_eigenvector, selected_eigenvalue 






## Note
### 𝕄_real and 𝕄_imaginary are not the real and imaginary part of 𝕄, but on the contrary the complex value of 𝕄 with the two different alignment framework

In [68]:
# use example

n = 10

S = generate_random_haar(n)

k1 = 0
k2 = 1
sigma_real, sigma_imaginary, 𝕄_real, 𝕄_imaginary, selected_eigenvector, selected_eigenvalue = run_experiment(S, k1, k2)
print(S)

[[ 1.12089416e-01+1.63457025e-01j  4.17660011e-04-2.76631478e-01j
   1.28039214e-01-5.08364222e-03j  4.20227049e-01+1.72232376e-01j
   2.69511860e-02+1.08346942e-01j  2.95670864e-01-1.58151575e-01j
   1.68430929e-01-3.92859900e-02j  1.11419275e-02-7.84166583e-02j
  -2.76840296e-01-5.18930463e-01j  3.67294254e-01+1.39997265e-01j]
 [-6.93178669e-03+3.44496302e-01j  2.21172018e-02-3.97826497e-01j
   2.23728567e-01+2.62837749e-01j -4.25559827e-01-1.36717303e-01j
   2.58281251e-01+8.32348446e-02j  1.29568556e-02-4.81938301e-02j
  -6.10250714e-02+1.55381747e-01j  3.88834630e-01-3.14770249e-04j
  -1.95335549e-01-1.08376563e-01j -2.38935430e-01-2.03491609e-01j]
 [ 1.09057654e-01-6.74105752e-02j -7.11924007e-03-2.20680539e-01j
  -1.13464720e-01+1.48354675e-01j  8.65886374e-02+2.07796489e-01j
  -2.71901283e-01-6.73185261e-02j -7.10721521e-01-2.36232617e-01j
  -1.39646417e-01-1.64379117e-01j  2.40104360e-01-1.08457307e-01j
   1.85751683e-01-2.07408197e-01j  8.52526342e-02-9.55706275e-02j]
 [ 6.50

In [59]:
print(sigma_real, sigma_imaginary, 𝕄_real, 𝕄_imaginary, selected_eigenvalue)
print("")
print(selected_eigenvector)

[ 1 -1  1 -1  1  1  1  1 -1 -1] [ 1  1 -1 -1 -1  1  1  1  1 -1] (0.14571629417839424-0.19623114290439386j) (-0.5083206560315411+0.2958728212228381j) (-0.31974359208208436-0.00011447941904599301j)

[ 0.14738034+0.14333318j -0.03296513+0.19037301j  0.04419917-0.09899804j
 -0.02187334-0.11746895j  0.37345321-0.20282938j  0.20898754+0.20479586j
  0.08355303+0.00339401j  0.2953512 +0.26383491j -0.22163086+0.60082338j
 -0.04868167-0.22767868j]


In [77]:
def check_unitary(S):
    """
    Return a Bool True/False if the matrix is unitary or not
    
    """
    out = True # initial assumption 
    tol = 10**(-5) # error tolerance
    for i in range(len(S)):
        count = 0
        for j in range(len(S)):
            count += np.dot(np.conjugate(S[i,j]), S[i,j])
        if count > 1+tol or count < 1-tol:
            out = False
    return out


In [83]:
# hand construction of a matrix which is not unitary
S = np.zeros((n, n), dtype=complex)
# print(S.shape)

# filling the matrix wrongfully

for i in range(n):
    for r in range(n):
        S[i, r] = np.random.randn() + 1j*np.random.randn()# this is not unitary but a complex number

check_unitary(S)

False

In [84]:
S = generate_random_haar(n)
check_unitary(S)

True

In [ ]:
def find_best_alignment(S):
"""
Find all the permanents and perform an experiment on all of them 
"""

    list_sigma_real, list_sigma_imaginary, list_𝕄_real, list_𝕄_imaginary, list_selected_eigenvector, list_selected_eigenvalue, couple_of_k = []
    for k1 in range(np.length(S)):
        for k2 in range(np.length(S)):
            sigma_real, sigma_imaginary, 𝕄_real, 𝕄_imaginary, selected_eigenvector, selected_eigenvalue = run_experiment(S, k1, k2)

            lista